# System operations (RQ-OPS): cost of serving and updating, per update policy

Thesis §4.5 / §5.5, Tables E.30–E.35, rebuilt on the post-fix runs (28 Nov 2026).

The thesis built these tables from a single monitor-attached run. Leak-free, the thesis
Signal 1/2 monitor never triggers an update, so that run alone has no update cost to show.
This notebook therefore puts **five update policies** side by side on the same CSCAS
windows (symbolic schema = Base(5) + mined, `baseline_split`):

| Policy | Run | Updates |
|---|---|---|
| never | temporal decay `20261128_095048` (gran 0.1/0.05) + `20261128_115921` (gran 0.02) | none after setup |
| always | rolling walk-forward `20261128_102658` | re-mine + refit every window |
| monitor: thesis | monitor_attached `20261125_122218` (Signal 1/2, PSI 0.1, cal 0.1, 3 windows) | triggered |
| monitor: feedback | monitor_attached `20261125_110537` (feedback precision → re-mine) | triggered |
| monitor: combined | monitor_attached `20261125_112254` (feedback → retrain, pattern validity → re-mine) | triggered |

**Windows compared:** the test windows W1…W(n−1) for every policy. That means h ≥ 1 for the
frozen and monitor runs (h0 is the source window's held-out split), and every step for the
walk-forward (step i is scored on W(i+1)).

**Update cost** excludes the one-time setup (mining + fit on the source window), which is the
same for every policy and reported separately.

**Caveat, always-retrain mining time:** the walk-forward run's mining steps were all cache
hits (the windows had been mined by an earlier run), so its `mining_wall_s` is not a real
measurement. For the always policy, mining time per update is therefore taken from the
**monitor runs' re-mine events at the same granularity and model**: same mining code, uncached,
on the same windows. Its fit time is measured. See `ALWAYS_MINING_SOURCE` below.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from thesis.paths import ROOT

EXP = ROOT / "artifacts" / "experiments"
FEATURE_SET = "symbolic"
MODELS = ["rf", "xgboost"]
GRANS = [0.02, 0.05, 0.1]

RUNS = {
    "never": [EXP / "temporal_decay/cscas/20261128_095048", EXP / "temporal_decay/cscas/20261128_115921"],
    "always": EXP / "rolling_walk_forward/cscas/20261128_102658",
    "monitor: thesis": EXP / "monitor_attached/cscas/20261125_122218",
    "monitor: feedback": EXP / "monitor_attached/cscas/20261125_110537",
    "monitor: combined": EXP / "monitor_attached/cscas/20261125_112254",
}
POLICY_ORDER = list(RUNS)
# Where the always policy's per-update mining time comes from (see the caveat above).
ALWAYS_MINING_SOURCE = ["monitor: feedback", "monitor: combined"]

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Load every run into one per-window frame

In [2]:
def _read_events(run_dir: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(run_dir / "events.csv")
    except pd.errors.EmptyDataError:  # the thesis trigger never fires leak-free
        return pd.DataFrame(columns=["granularity", "model", "executed", "action",
                                     "t_mining_s", "t_retrain_s", "t_deploy_s",
                                     "mining_cpu_s", "retrain_cpu_s", "churn_frac"])


KEEP = ["policy", "granularity", "model", "window", "f1", "tp", "fp", "fn", "n_alert_groups",
        "n_alerts_in", "n_groups_escalated", "n_groups_suppressed", "n_alerts_escalated",
        "n_alerts_suppressed", "fast_route_wall_s", "fast_route_cpu_s"]

frames, events, setup = [], [], []

# never: frozen per-horizon rows, h >= 1
td = pd.concat([pd.read_csv(p / "per_horizon_results.csv", low_memory=False) for p in RUNS["never"]])
td = td[td["feature_set"].eq(FEATURE_SET)]
setup.append(td.groupby(["granularity", "model"])[["mining_wall_s", "fit_wall_s", "mining_cpu_s", "fit_cpu_s"]].first()
             .assign(mining_cache_hit=td.groupby(["granularity", "model"])["mining_cache_hit"].first()))
td = td[td["horizon_window_index"] >= 1].assign(policy="never", window=lambda d: d["horizon_window_index"])
frames.append(td)

# always: walk-forward steps (step i scored on W(i+1)); one update per step
wf = pd.read_csv(RUNS["always"] / "per_step_results.csv", low_memory=False)
wf = wf[wf["feature_set"].eq(FEATURE_SET)].assign(policy="always", window=lambda d: d["step_i"] + 1)
frames.append(wf)
always_fit = wf[["granularity", "model", "window", "fit_wall_s", "fit_cpu_s", "mining_wall_s", "mining_cache_hit"]]

# monitor runs: per-horizon rows h >= 1, plus their executed events
for pol in POLICY_ORDER[2:]:
    ph = pd.read_csv(RUNS[pol] / "per_horizon_results.csv", low_memory=False)
    ph = ph[ph["feature_set"].eq(FEATURE_SET) & (ph["horizon_window_index"] >= 1)]
    frames.append(ph.assign(policy=pol, window=lambda d: d["horizon_window_index"]))
    ev = _read_events(RUNS[pol])
    ev = ev[ev["executed"].astype(bool)] if len(ev) else ev
    events.append(ev.assign(policy=pol))

win = pd.concat([f[KEEP] for f in frames], ignore_index=True)
win = win[win["granularity"].isin(GRANS) & win["model"].isin(MODELS)]
ev_all = pd.concat(events, ignore_index=True)
setup = pd.concat(setup)

print("windows per (policy, gran, model):")
print(win.groupby(["policy", "granularity", "model"]).size().unstack(["granularity"]).reindex(POLICY_ORDER, level=0))
print("\nalways-retrain mining cache hits:", always_fit["mining_cache_hit"].mean())

windows per (policy, gran, model):
granularity                0.02  0.05  0.10
policy            model                    
never             rf         50    20    10
                  xgboost    50    20    10
always            rf         50    20    10
                  xgboost    50    20    10
monitor: thesis   rf         50    20    10
                  xgboost    50    20    10
monitor: feedback rf         50    20    10
                  xgboost    50    20    10
monitor: combined rf         50    20    10
                  xgboost    50    20    10

always-retrain mining cache hits: 1.0


/var/folders/69/6nn2sm8d7qq0p98t_qgp1ckr0000gn/T/ipykernel_45577/3375447218.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ev_all = pd.concat(events, ignore_index=True)


## 2. Update cost per policy

$L_i = t_\text{mining} + t_\text{retrain} + t_\text{deploy}$ per executed update. For the always
policy, $t_\text{retrain}$ is the measured fit time per step and $t_\text{mining}$ the mean
uncached re-mine time from the monitor runs at the same granularity and model.

In [3]:
ev = ev_all.copy()
ev["is_remine"] = ev["action"].eq("REMINE_AND_RETRAIN")
ev["L_s"] = ev[["t_mining_s", "t_retrain_s", "t_deploy_s"]].fillna(0).sum(axis=1)
ev["cpu_s"] = ev[["mining_cpu_s", "retrain_cpu_s"]].fillna(0).sum(axis=1)

# Uncached mining time per re-mine, from the monitor runs, by (gran, model).
mine_ref = (ev[ev["policy"].isin(ALWAYS_MINING_SOURCE) & ev["is_remine"]]
            .groupby(["granularity", "model"])[["t_mining_s", "mining_cpu_s"]].mean())
print("uncached mining time per re-mine (s), from the monitor runs:")
print(mine_ref.round(2))

aw = always_fit.join(mine_ref, on=["granularity", "model"])
aw = aw.assign(policy="always", is_remine=True,
               L_s=aw["t_mining_s"] + aw["fit_wall_s"],
               cpu_s=aw["mining_cpu_s"] + aw["fit_cpu_s"])
updates = pd.concat([ev[["policy", "granularity", "model", "is_remine", "L_s", "cpu_s"]],
                     aw[["policy", "granularity", "model", "is_remine", "L_s", "cpu_s"]]], ignore_index=True)
updates = updates[updates["granularity"].isin(GRANS) & updates["model"].isin(MODELS)]

uncached mining time per re-mine (s), from the monitor runs:
                     t_mining_s  mining_cpu_s
granularity model                            
0.02        rf             0.88          1.52
            xgboost        1.69          2.05
0.05        rf             6.26         16.01
            xgboost        6.68         16.77
0.10        rf            14.29         43.85
            xgboost       12.16         20.06


## 3. Tables

In [4]:
def per_cfg(g: pd.DataFrame) -> pd.Series:
    fr = g["fast_route_wall_s"].dropna()
    thr = (g["n_alerts_in"] / g["fast_route_wall_s"].replace(0, np.nan) * 60).dropna()
    esc, sup = g["n_groups_escalated"].sum(), g["n_groups_suppressed"].sum()
    return pd.Series({
        "n_windows": len(g),
        "f1_mean": g["f1"].mean(), "f1_std": g["f1"].std(),
        "fr_mean_s": fr.mean(), "fr_p95_s": fr.quantile(0.95), "fr_total_s": fr.sum(),
        "in_mean": g["n_alerts_in"].mean(), "in_max": g["n_alerts_in"].max(),
        "throughput_per_min": thr.mean(),
        "N": g["n_alerts_in"].sum(), "G": g["n_alert_groups"].sum(),
        "R_escalated": esc / g["n_alert_groups"].sum(),
        "ARR": 1 - g["n_alerts_escalated"].sum() / g["n_alerts_in"].sum(),
        "FNR_suppressed": g["fn"].sum() / sup if sup else np.nan,
        "precision_escalated": g["tp"].sum() / esc if esc else np.nan,
    })


keys = ["policy", "granularity", "model"]
serve = win.groupby(keys).apply(per_cfg, include_groups=False)
upd = updates.groupby(keys).agg(n_updates=("L_s", "size"), n_remines=("is_remine", "sum"),
                                L_mean_s=("L_s", "mean"), L_total_s=("L_s", "sum"),
                                cpu_mean_s=("cpu_s", "mean"), cpu_total_s=("cpu_s", "sum"))
T = serve.join(upd).reset_index()
T[["n_updates", "n_remines", "L_total_s", "cpu_total_s"]] = T[["n_updates", "n_remines", "L_total_s", "cpu_total_s"]].fillna(0)
T["adaptation_ratio"] = T["L_total_s"] / T["fr_total_s"]
T["policy"] = pd.Categorical(T["policy"], POLICY_ORDER, ordered=True)
T = T.sort_values(["granularity", "model", "policy"]).reset_index(drop=True)

print("One-time setup on the source window (same for every policy):")
print(setup.round(2))

One-time setup on the source window (same for every policy):
                     mining_wall_s  fit_wall_s  mining_cpu_s  fit_cpu_s mining_cache_hit
granularity model                                                                       
0.02        rf                0.01        0.67          0.01       4.98             True
            xgboost           0.22        3.80          0.22      10.58             True
0.05        rf                5.37        0.96          5.37      10.17             True
            xgboost           5.41        3.52          5.41      15.85             True
0.10        rf                5.34        0.95          5.34      10.08            False
            xgboost           5.45        4.79          5.45      17.47             True


In [5]:
cost = T[["granularity", "model", "policy", "f1_mean", "n_updates", "n_remines", "L_mean_s",
          "L_total_s", "fr_mean_s", "fr_p95_s", "fr_total_s", "adaptation_ratio"]]
cost.round(3)

,granularity,model,policy,f1_mean,n_updates,n_remines,L_mean_s,L_total_s,fr_mean_s,fr_p95_s,fr_total_s,adaptation_ratio
0,0.02,rf,never,0.846,0.0,0.0,NaN,0.000,1.300,2.036,64.979,0.000
1,0.02,rf,always,0.928,50.0,50.0,2.856,142.796,4.712,8.397,235.617,0.606
2,0.02,rf,monitor: thesis,0.844,0.0,0.0,NaN,0.000,1.564,4.971,78.207,0.000
3,0.02,rf,monitor: feedback,0.894,11.0,11.0,1.732,19.054,1.379,3.312,68.933,0.276
4,0.02,rf,monitor: combined,0.909,11.0,1.0,1.060,11.660,1.837,5.492,91.873,0.127
5,0.02,xgboost,never,0.882,0.0,0.0,NaN,0.000,0.941,1.107,47.072,0.000
6,0.02,xgboost,always,0.922,50.0,50.0,7.432,371.580,2.278,3.807,113.878,3.263
7,0.02,xgboost,monitor: thesis,0.832,0.0,0.0,NaN,0.000,1.781,4.295,89.062,0.000
8,0.02,xgboost,monitor: feedback,0.878,5.0,5.0,6.127,30.635,1.118,1.997,55.921,0.548
9,0.02,xgboost,monitor: combined,0.900,12.0,3.0,5.713,68.555,1.333,2.389,66.644,1.029


**Reading the timing columns:** fast-route wall time comes from runs made at different times on the shared DGX, with different background load. Compare it within a policy (flat between updates?), not across policies. For example, the thesis-monitor run never updates and has the same model quality as the frozen run, but its fast route is ~2–3× slower: that difference is machine load, not the policy. Update counts, $L_i$ and F1 are the cross-policy quantities; the adaptation ratio mixes both, so treat it as an order of magnitude.

In [6]:
thru = T[["granularity", "model", "policy", "in_mean", "in_max", "throughput_per_min", "ARR"]]
thru.round(3)

,granularity,model,policy,in_mean,in_max,throughput_per_min,ARR
0,0.02,rf,never,211603.32,367416.0,1.023001e+07,0.993
1,0.02,rf,always,211603.32,367416.0,4.469155e+06,0.995
2,0.02,rf,monitor: thesis,211603.32,367416.0,1.266826e+07,0.993
3,0.02,rf,monitor: feedback,211603.32,367416.0,1.278586e+07,0.995
4,0.02,rf,monitor: combined,211603.32,367416.0,1.083923e+07,0.995
5,0.02,xgboost,never,211603.32,367416.0,1.368622e+07,0.992
6,0.02,xgboost,always,211603.32,367416.0,7.092964e+06,0.994
7,0.02,xgboost,monitor: thesis,211603.32,367416.0,1.035468e+07,0.993
8,0.02,xgboost,monitor: feedback,211603.32,367416.0,1.440341e+07,0.995
9,0.02,xgboost,monitor: combined,211603.32,367416.0,1.217766e+07,0.994


In [7]:
work = T[["granularity", "model", "policy", "N", "G", "R_escalated", "ARR", "FNR_suppressed",
          "precision_escalated", "f1_mean"]]
work.round(4)

,granularity,model,policy,N,G,R_escalated,ARR,FNR_suppressed,precision_escalated,f1_mean
0,0.02,rf,never,10580166.0,1255792.0,0.0150,0.9930,0.0025,0.8548,0.8462
1,0.02,rf,always,10580166.0,1255792.0,0.0149,0.9946,0.0013,0.9367,0.9280
2,0.02,rf,monitor: thesis,10580166.0,1255792.0,0.0151,0.9932,0.0025,0.8501,0.8439
3,0.02,rf,monitor: feedback,10580166.0,1255792.0,0.0141,0.9946,0.0021,0.9353,0.8941
4,0.02,rf,monitor: combined,10580166.0,1255792.0,0.0142,0.9948,0.0019,0.9453,0.9088
5,0.02,xgboost,never,10580166.0,1255792.0,0.0154,0.9923,0.0017,0.8804,0.8824
6,0.02,xgboost,always,10580166.0,1255792.0,0.0154,0.9938,0.0011,0.9196,0.9223
7,0.02,xgboost,monitor: thesis,10580166.0,1255792.0,0.0138,0.9935,0.0031,0.8820,0.8321
8,0.02,xgboost,monitor: feedback,10580166.0,1255792.0,0.0149,0.9946,0.0020,0.8916,0.8781
9,0.02,xgboost,monitor: combined,10580166.0,1255792.0,0.0154,0.9939,0.0015,0.8983,0.9000


## 4. LaTeX (inline, copy into the thesis)

In [8]:
def _f(v, nd=2, pct=False, thousands=False):
    if pd.isna(v):
        return "--"
    if pct:
        return f"{100 * v:.{nd}f}\\%"
    if thousands:
        return f"{v:,.0f}"
    return f"{v:.{nd}f}"


def latex_table(df, cols, header, caption, label):
    lines = [r"\begin{table}[h]", r"\centering", r"\footnotesize",
             r"\begin{tabular}{ll" + "l" + "r" * len(cols) + "}", r"\toprule",
             r"\textbf{Gran} & \textbf{Model} & \textbf{Policy} & " + " & ".join(header) + r" \\", r"\midrule"]
    prev = (None, None)
    for _, r in df.iterrows():
        g = f"{r['granularity']:g}" if r["granularity"] != prev[0] else ""
        m = r["model"] if (r["granularity"], r["model"]) != prev else ""
        if r["granularity"] != prev[0] and prev[0] is not None:
            lines.append(r"\midrule")
        prev = (r["granularity"], r["model"])
        lines.append(f"{g} & {m} & {r['policy']} & " + " & ".join(fmt(r[c]) for c, fmt in cols) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", rf"\caption{{{caption}}}", rf"\label{{{label}}}", r"\end{table}"]
    print("\n".join(lines))


latex_table(
    T,
    [("f1_mean", lambda v: _f(v, 3)), ("n_updates", lambda v: f"{int(v)}"), ("n_remines", lambda v: f"{int(v)}"),
     ("L_mean_s", _f), ("L_total_s", lambda v: _f(v, 0)), ("fr_mean_s", _f), ("fr_p95_s", _f),
     ("adaptation_ratio", lambda v: _f(v, 1, pct=True))],
    ["F1", "\\# upd.", "\\# re-mine", "$L_i$ mean (s)", "$\\sum L_i$ (s)", "Fast route mean (s)",
     "Fast route p95 (s)", "Adapt.\\ ratio"],
    "Serving vs.\\ updating cost per update policy (CSCAS, symbolic schema, test windows $W_1$--$W_{n-1}$). "
    "Fast route = batched encode+predict wall time per window. $L_i = t_\\text{mining}+t_\\text{retrain}+t_\\text{deploy}$ "
    "per executed update, excluding the one-time setup. Adaptation ratio = $\\sum L_i$ / total fast-route time. "
    "Always-retrain mining time per update is the uncached re-mine time measured in the monitor runs.",
    "tab:sysops-cost-policies",
)

\begin{table}[h]
\centering
\footnotesize
\begin{tabular}{lllrrrrrrrr}
\toprule
\textbf{Gran} & \textbf{Model} & \textbf{Policy} & F1 & \# upd. & \# re-mine & $L_i$ mean (s) & $\sum L_i$ (s) & Fast route mean (s) & Fast route p95 (s) & Adapt.\ ratio \\
\midrule
0.02 & rf & never & 0.846 & 0 & 0 & -- & 0 & 1.30 & 2.04 & 0.0\% \\
 &  & always & 0.928 & 50 & 50 & 2.86 & 143 & 4.71 & 8.40 & 60.6\% \\
 &  & monitor: thesis & 0.844 & 0 & 0 & -- & 0 & 1.56 & 4.97 & 0.0\% \\
 &  & monitor: feedback & 0.894 & 11 & 11 & 1.73 & 19 & 1.38 & 3.31 & 27.6\% \\
 &  & monitor: combined & 0.909 & 11 & 1 & 1.06 & 12 & 1.84 & 5.49 & 12.7\% \\
 & xgboost & never & 0.882 & 0 & 0 & -- & 0 & 0.94 & 1.11 & 0.0\% \\
 &  & always & 0.922 & 50 & 50 & 7.43 & 372 & 2.28 & 3.81 & 326.3\% \\
 &  & monitor: thesis & 0.832 & 0 & 0 & -- & 0 & 1.78 & 4.29 & 0.0\% \\
 &  & monitor: feedback & 0.878 & 5 & 5 & 6.13 & 31 & 1.12 & 2.00 & 54.8\% \\
 &  & monitor: combined & 0.900 & 12 & 3 & 5.71 & 69 & 1.33 & 2.39 & 102.9\% \\

In [9]:
latex_table(
    T,
    [("in_mean", lambda v: _f(v, thousands=True)), ("in_max", lambda v: _f(v, thousands=True)),
     ("throughput_per_min", lambda v: _f(v, thousands=True)), ("ARR", lambda v: _f(v, 1, pct=True))],
    ["Mean In($t$)", "Max In($t$)", "Throughput (alerts/min)", "Reduction rate"],
    "Throughput and alert reduction per update policy. In($t$) = raw alerts in window $t$; throughput = "
    "In($t$) / fast-route wall time; reduction rate = share of raw alerts not escalated.",
    "tab:sysops-throughput-policies",
)

\begin{table}[h]
\centering
\footnotesize
\begin{tabular}{lllrrrr}
\toprule
\textbf{Gran} & \textbf{Model} & \textbf{Policy} & Mean In($t$) & Max In($t$) & Throughput (alerts/min) & Reduction rate \\
\midrule
0.02 & rf & never & 211,603 & 367,416 & 10,230,013 & 99.3\% \\
 &  & always & 211,603 & 367,416 & 4,469,155 & 99.5\% \\
 &  & monitor: thesis & 211,603 & 367,416 & 12,668,260 & 99.3\% \\
 &  & monitor: feedback & 211,603 & 367,416 & 12,785,856 & 99.5\% \\
 &  & monitor: combined & 211,603 & 367,416 & 10,839,227 & 99.5\% \\
 & xgboost & never & 211,603 & 367,416 & 13,686,216 & 99.2\% \\
 &  & always & 211,603 & 367,416 & 7,092,964 & 99.4\% \\
 &  & monitor: thesis & 211,603 & 367,416 & 10,354,680 & 99.3\% \\
 &  & monitor: feedback & 211,603 & 367,416 & 14,403,409 & 99.5\% \\
 &  & monitor: combined & 211,603 & 367,416 & 12,177,657 & 99.4\% \\
\midrule
0.05 & rf & never & 529,008 & 785,279 & 10,999,067 & 99.3\% \\
 &  & always & 529,008 & 785,279 & 5,718,902 & 99.3\% \\
 &  & monit

In [10]:
latex_table(
    T,
    [("N", lambda v: _f(v, thousands=True)), ("G", lambda v: _f(v, thousands=True)),
     ("R_escalated", lambda v: _f(v, 2, pct=True)), ("ARR", lambda v: _f(v, 2, pct=True)),
     ("FNR_suppressed", lambda v: _f(v, 2, pct=True)), ("precision_escalated", lambda v: _f(v, 1, pct=True))],
    ["$N$", "$G$", "$R_\\text{escalated}$", "ARR", "FNR$_\\text{suppr.}$", "Precision$_\\text{esc.}$"],
    "Workload reduction per update policy. $N$ = raw alerts, $G$ = alert groups over the test windows; "
    "$R_\\text{escalated}$ = escalated groups / $G$; ARR = alert reduction rate; FNR$_\\text{suppr.}$ = attack groups "
    "among suppressed groups; Precision$_\\text{esc.}$ = attack groups among escalated groups.",
    "tab:sysops-workload-policies",
)

\begin{table}[h]
\centering
\footnotesize
\begin{tabular}{lllrrrrrr}
\toprule
\textbf{Gran} & \textbf{Model} & \textbf{Policy} & $N$ & $G$ & $R_\text{escalated}$ & ARR & FNR$_\text{suppr.}$ & Precision$_\text{esc.}$ \\
\midrule
0.02 & rf & never & 10,580,166 & 1,255,792 & 1.50\% & 99.30\% & 0.25\% & 85.5\% \\
 &  & always & 10,580,166 & 1,255,792 & 1.49\% & 99.46\% & 0.13\% & 93.7\% \\
 &  & monitor: thesis & 10,580,166 & 1,255,792 & 1.51\% & 99.32\% & 0.25\% & 85.0\% \\
 &  & monitor: feedback & 10,580,166 & 1,255,792 & 1.41\% & 99.46\% & 0.21\% & 93.5\% \\
 &  & monitor: combined & 10,580,166 & 1,255,792 & 1.42\% & 99.48\% & 0.19\% & 94.5\% \\
 & xgboost & never & 10,580,166 & 1,255,792 & 1.54\% & 99.23\% & 0.17\% & 88.0\% \\
 &  & always & 10,580,166 & 1,255,792 & 1.54\% & 99.38\% & 0.11\% & 92.0\% \\
 &  & monitor: thesis & 10,580,166 & 1,255,792 & 1.38\% & 99.35\% & 0.31\% & 88.2\% \\
 &  & monitor: feedback & 10,580,166 & 1,255,792 & 1.49\% & 99.46\% & 0.20\% & 89.2\% \\
 &  & mo